In [1]:
import json
from pathlib import Path

import numpy as np
import polars as pl

from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [ ]:
split = Path("../competition/train")
feed = (
    pl.scan_parquet(str(split / "impressions/part-*.parquet"))
    .head(1)
    .collect()
    .row(0, named=True)
)

embedding_dir = split / "item_embeddings"
meta = json.loads((embedding_dir / "data.meta").read_text())

embeddings = np.memmap(
    embedding_dir / "data.memmap",
    dtype=meta["dtype"],
    mode="r",
    shape=tuple(meta["shape"]),
    order=meta["order"],
)

indices = np.asarray(feed["item_id__embedding_idx"], dtype=np.int64)
present = indices >= 0
candidate_vectors = np.zeros((len(indices), 128), dtype=np.float32)
candidate_vectors[present] = embeddings[indices[present]].astype(np.float32)
user_vector = np.asarray(feed["user_embedding"], dtype=np.float32)

In [38]:
data_path = Path('../competition/train')
train_data = pl.scan_parquet(data_path / 'impressions/part-*.parquet')
# train_data.head(5).collect()['f1'].shape
train_data.head(1).collect()

feed_id,timestamp,user_id,category_id,f1,f2,f3,f4,f5,pos,target,item_id__embedding_idx,item_id,user_embedding,history__item_id,history__category_id,history__event_type,history__event_source,history__item_id__embedding_idx
i32,i32,i32,list[i8],list[f32],list[f32],list[f32],list[f32],list[f32],list[i16],list[i8],list[i32],list[i32],list[i8],list[i32],list[i8],list[i8],list[i8],list[i32]
2304337,1789413888,56184,"[3, 13, … 13]","[0.283893, 0.736484, … 0.74508]","[0.093346, 0.407478, … 0.790085]","[0.238096, 0.708231, … 0.708231]","[0.645679, 0.940511, … 0.613674]","[0.675874, 0.13994, … 0.896418]","[42, 14, … 40]","[0, 0, … 0]","[7939845, 11581808, … 80453627]","[73328213, 47564870, … 85697676]","[2, 6, … -19]","[74845037, 38544617, … 59698341]","[13, 13, … 3]","[1, 1, … 1]","[1, 1, … 1]","[35257591, 69350219, … -1]"


In [30]:
emb_dir = data_path / 'item_embeddings'

meta = json.loads((emb_dir/'data.meta').read_text())

In [34]:
embeddings = np.memmap(
    emb_dir/'data.memmap',
    dtype=meta['dtype'],
    mode='r',
    shape=tuple(meta['shape']),
    order=meta['order']
)

In [44]:
idx = np.asarray(feed["item_id__embedding_idx"], dtype=np.int64)
pras = idx > 0
candidate_vectors = np.zeros((len(indices), 128), dtype=np.float32)
candidate_vectors[present] = embeddings[indices[present]].astype(np.float32)
user_vector = np.asarray(feed["user_embedding"], dtype=np.float32)

In [36]:
embeddings.shape

(81557944, 128)

In [48]:
candidate_vectors

array([[ -9.,  14.,  18., ...,  -2., -27.,   5.],
       [-11., -30.,  31., ...,  25.,  14.,   3.],
       [ 23., -27.,  81., ..., -22.,   6.,  -2.],
       ...,
       [ -2., -23.,   0., ...,  54.,  10.,   8.],
       [ -1., -21.,   7., ...,  27.,  14.,   5.],
       [ -4., -24.,   1., ...,  23.,  -4.,   5.]], dtype=float32)

In [3]:
user_vector.shape

(128,)

In [4]:
candidate_vectors.shape

(72, 128)

In [5]:
feed.keys()

dict_keys(['feed_id', 'timestamp', 'user_id', 'category_id', 'f1', 'f2', 'f3', 'f4', 'f5', 'pos', 'target', 'item_id__embedding_idx', 'item_id', 'user_embedding', 'history__item_id', 'history__category_id', 'history__event_type', 'history__event_source', 'history__item_id__embedding_idx'])

In [ ]:
import argparse
import json
from pathlib import Path

import lightgbm as lgb
import numpy as np
import polars as pl

K_HIST = 32  # сколько последних событий истории берём для сходства эмбеддингов
EPS = 1e-6


def load_emb(split_dir: Path):
    d = split_dir / "item_embeddings"
    meta = json.loads((d / "data.meta").read_text())
    return np.memmap(d / "data.memmap", dtype=meta["dtype"], mode="r",
                     shape=tuple(meta["shape"]), order=meta["order"])


def unit(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + EPS)


def zscore(x):
    m = np.nanmean(x, axis=0) if np.isfinite(x).any() else 0.0
    s = np.nanstd(x, axis=0) if np.isfinite(x).any() else 1.0
    return (x - m) / (s + EPS)


def feed_features(row, emb, use_pos, train):
    ids = np.asarray(row["item_id"])
    cat = np.asarray(row["category_id"])
    M = len(ids)

    # --- числовые признаки + относительные внутри ленты
    F = np.stack([np.asarray(row[f"f{i}"], dtype=np.float32) for i in range(1, 6)], 1)
    Fz = zscore(F)
    Fr = F.argsort(0).argsort(0) / max(M - 1, 1)

    # --- эмбеддинги кандидатов
    cidx = np.asarray(row["item_id__embedding_idx"], dtype=np.int64)
    present = cidx >= 0
    C = np.zeros((M, 128), dtype=np.float32)
    if present.any():
        C[present] = emb[cidx[present]].astype(np.float32)
    C = unit(C)
    u = unit(np.asarray(row["user_embedding"], dtype=np.float32))

    cu = C @ u
    cu[~present] = np.nan
    cu_z = zscore(cu[:, None])[:, 0]
    cu_rank = np.where(present, np.argsort(np.argsort(-np.nan_to_num(cu, nan=-9))) / max(M - 1, 1), np.nan)

    # --- сходство с историей
    h_idx = np.asarray(row["history__item_id__embedding_idx"], dtype=np.int64)[:K_HIST]
    h_idx = h_idx[h_idx >= 0]
    if len(h_idx) and present.any():
        H = unit(emb[h_idx].astype(np.float32))
        S = C @ H.T
        s_max, s_mean, s_last5 = S.max(1), S.mean(1), S[:, :5].mean(1)
        for a in (s_max, s_mean, s_last5):
            a[~present] = np.nan
    else:
        s_max = s_mean = s_last5 = np.full(M, np.nan, dtype=np.float32)
    s_mean_z = zscore(s_mean[:, None])[:, 0]

    # --- категории и товары из истории
    hc = np.asarray(row["history__category_id"])
    hi = np.asarray(row["history__item_id"])
    H_len = len(hc)
    if H_len:
        same = hc[None, :] == cat[:, None]
        cat_share = same.mean(1)
        cat_share20 = same[:, :20].mean(1)
        in_hist = (hi[None, :] == ids[:, None]).sum(1)
    else:
        cat_share = cat_share20 = np.zeros(M)
        in_hist = np.zeros(M)

    cols = [
        F, Fz, Fr,
        cat[:, None], np.full((M, 1), M), np.full((M, 1), H_len),
        cu[:, None], cu_z[:, None], cu_rank[:, None],
        s_max[:, None], s_mean[:, None], s_last5[:, None], s_mean_z[:, None],
        cat_share[:, None], cat_share20[:, None], in_hist[:, None],
        present[:, None],
    ]
    if use_pos:
        if train:
            pos = np.asarray(row["pos"], dtype=np.float32)
            pos = np.where(pos == None, np.nan, pos)  # noqa: E711 (пропуски)
        else:
            pos = np.zeros(M, dtype=np.float32)  # константа на инференсе
        cols.append(pos[:, None])
    X = np.hstack([c.astype(np.float32) for c in cols])
    y = np.asarray(row["target"], dtype=np.int8) if "target" in row else None
    return X, y, ids


def build(split_dir: Path, max_feeds, use_pos, train):
    lf = pl.scan_parquet(str(split_dir / "impressions/part-*.parquet"))
    if max_feeds:
        lf = lf.head(max_feeds)
    df = lf.collect()
    emb = load_emb(split_dir)
    Xs, ys, groups, fids, iids = [], [], [], [], []
    for row in df.iter_rows(named=True):
        X, y, ids = feed_features(row, emb, use_pos, train)
        Xs.append(X)
        groups.append(len(ids))
        fids.append(np.full(len(ids), row["feed_id"], dtype=np.int32))
        iids.append(ids.astype(np.int32))
        if y is not None:
            ys.append(y)
    return (np.vstack(Xs), np.concatenate(ys) if ys else None,
            np.asarray(groups), np.concatenate(fids), np.concatenate(iids))


def ndcg_at_k(scores, y, groups, k=30):
    """NDCG@k как в описании: ничьи -> негативы выше; нет позитивов -> 1."""
    disc = 1.0 / np.log2(np.arange(2, k + 2))
    out, start = [], 0
    for g in groups:
        s, t = scores[start:start + g], y[start:start + g]
        start += g
        if t.sum() == 0:
            out.append(1.0)
            continue
        order = np.lexsort((t, -s))  # score по убыванию, при равенстве target=0 раньше
        top = t[order][:k]
        dcg = (top * disc[:len(top)]).sum()
        ideal = np.sort(t)[::-1][:k]
        idcg = (ideal * disc[:len(ideal)]).sum()
        out.append(dcg / idcg)
    return float(np.mean(out))


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", default="submission.parquet")
    ap.add_argument("--max-train-feeds", type=int, default=100_000)
    ap.add_argument("--max-val-feeds", type=int, default=None)
    ap.add_argument("--pos-trick", action="store_true")
    args = ap.parse_args(argv)
    root = Path(args.data)

    print("features: train...")
    Xtr, ytr, gtr, _, _ = build(root / "train", args.max_train_feeds, args.pos_trick, True)
    print("features: val...")
    Xva, yva, gva, _, _ = build(root / "val", args.max_val_feeds, args.pos_trick, False)

    model = lgb.LGBMRanker(
        objective="lambdarank",
        n_estimators=2000,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=50,
        subsample=0.8, subsample_freq=1,
        colsample_bytree=0.8,
        lambdarank_truncation_level=30,
        verbose=-1,
    )
    model.fit(
        Xtr, ytr, group=gtr,
        eval_set=[(Xva, yva)], eval_group=[gva], eval_at=[30],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(50)],
    )
    print("val NDCG@30:", ndcg_at_k(model.predict(Xva), yva, gva))

    print("features: test...")
    Xte, _, _, fte, ite = build(root / "test", None, args.pos_trick, False)
    pred = pl.DataFrame({
        "feed_id": fte, "item_id": ite,
        "score": model.predict(Xte).astype(np.float32),
    })
    sample = pl.read_parquet(root / "sample_submission.parquet").select("feed_id", "item_id")
    sub = sample.join(pred, on=["feed_id", "item_id"], how="left")
    assert sub["score"].null_count() == 0, "есть пары без предсказаний"
    assert sub.height == sample.height
    sub.write_parquet(args.out)
    print("saved", args.out, sub.shape)


# if __name__ == "__main__":
#     main()

In [7]:
main([
    "--data", r"../competition",
    "--out", "submission.parquet",
    "--max-train-feeds", "1000",
    "--pos-trick",
])

features: train...
features: val...


c:\Users\Lenovo\Desktop\AvitoMLCUP\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[50]	valid_0's ndcg@30: 0.266008
[100]	valid_0's ndcg@30: 0.265753
Early stopping, best iteration is:
[43]	valid_0's ndcg@30: 0.268907
val NDCG@30: 0.26890655185983103
features: test...
saved submission.parquet (2904806, 3)


In [18]:
test_feed = pl.scan_parquet('submission.parquet').collect()
test_feed.sort('score', descending=True)

feed_id,item_id,score
i32,i32,f32
1221337,86952716,1.156389
2192136,108579461,1.149382
621987,34781064,1.126063
2204112,51349898,1.108192
2912351,57868825,1.087045
…,…,…
587741,86746748,-1.880416
2718468,29120968,-1.882913
2740900,90045677,-1.887004
